In [1]:
import sys
sys.path.append('../')

In [2]:
# Load models.
import torch
from genwarp import GenWarp

# ZoeDepth
zoedepth = torch.hub.load(
    '../extern/ZoeDepth',
    'ZoeD_N',
    source='local',
    pretrained=True,
    trust_repo=True
).to('cuda')

# GenWarp
genwarp_cfg = dict(
    pretrained_model_path='../checkpoints',
    checkpoint_name='multi1',
    half_precision_weights=True
)
genwarp_nvs = GenWarp(cfg=genwarp_cfg)

img_size [384, 512]


Using cache found in /home/newview/.cache/torch/hub/intel-isl_MiDaS_master
/home/newview/.local/lib/python3.10/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3483.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Params passed to Resize transform:
	width:  512
	height:  384
	resize_target:  True
	keep_aspect_ratio:  True
	ensure_multiple_of:  32
	resize_method:  minimal
Using pretrained resource url::https://github.com/isl-org/ZoeDepth/releases/download/v1.0/ZoeD_M12_N.pt
Loaded successfully
Loading GenWarp...


/home/newview/.local/lib/python3.10/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
/home/newview/.local/lib/python3.10/site-packages/diffusers/models/lora.py:306: FutureWarning: `LoRACompatibleConv` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleConv` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleConv", "1.0.0", deprecation_message)


Loaded GenWarp.


In [ ]:
import os
import gc
import traceback
import random
from os.path import basename, splitext, join
from glob import glob

import numpy as np
from PIL import Image

import torch
import torch.nn.functional as F
from torchvision.transforms.functional import to_tensor, to_pil_image

from genwarp.ops import camera_lookat, get_projection_matrix, sph2cart, focal_length_to_fov
from extern.ZoeDepth.zoedepth.utils.misc import colorize

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------
image_dir = "../es_100_ref"
out_dir = "../es_100_ref_high"
os.makedirs(out_dir, exist_ok=True)

# Fixed azimuth; elevation & radius randomized per image
azi_deg = 0.0
ELE_OPTIONS = [11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
RADIUS_OPTIONS = [0.31, 0.32, 0.33, 0.34, 0.35, 0.36, 0.37, 0.38, 0.39, 0.40]
#  [6, 7, 8, 9, 10]
#  [1, 2, 3, 4, 5]
#  [11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
#  [0.31, 0.32, 0.33, 0.34, 0.35, 0.36, 0.37, 0.38, 0.39, 0.40]
#  [0.21, 0.22, 0.23, 0.24, 0.25, 0.26, 0.27, 0.28, 0.29, 0.30]
#  [0.11, 0.12, 0.13, 0.14, 0,15, 0.16, 0.17, 0.18, 0.19, 0.20]


focal_length_mm = 19
near, far = 0.01, 100
exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = True  # set False if you want stricter bit-level repeatability

# HELPERS
def get_first_n_images(folder: str, n: int = 30):
    files = []
    for e in exts:
        files.extend(glob(join(folder, e)))
    files = sorted(files)  # deterministic file order
    return files[:n]

def cleanup_vars(local_vars: dict, names):
    for n in names:
        if n in local_vars:
            try:
                del local_vars[n]
            except Exception:
                pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def pad_to_multiple(pil: Image.Image, mult: int = 8, fill=(0, 0, 0)):
    """
    Pads image (letterbox) to the next multiple of `mult` for both H and W.
    Returns: padded_pil, (left, top, right, bottom) pad amounts.
    """
    w, h = pil.size
    new_w = int(np.ceil(w / mult) * mult)
    new_h = int(np.ceil(h / mult) * mult)

    pad_w = new_w - w
    pad_h = new_h - h
    left = pad_w // 2
    right = pad_w - left
    top = pad_h // 2
    bottom = pad_h - top

    if pad_w == 0 and pad_h == 0:
        return pil, (0, 0, 0, 0)

    canvas = Image.new("RGB", (new_w, new_h), fill)
    canvas.paste(pil, (left, top))
    return canvas, (left, top, right, bottom)

def unpad_pil(pil: Image.Image, pad):
    left, top, right, bottom = pad
    if (left, top, right, bottom) == (0, 0, 0, 0):
        return pil
    w, h = pil.size
    return pil.crop((left, top, w - right, h - bottom))

def colorize_to_pil(depth_1hw: torch.Tensor) -> Image.Image:
    if isinstance(depth_1hw, torch.Tensor):
        d = depth_1hw.detach()
        if d.ndim == 3 and d.shape[0] == 1:
            d = d[0]
        d = d.float().cpu()
    else:
        d = depth_1hw
        if d.ndim == 3 and d.shape[0] == 1:
            d = d[0]
        d = d.astype(np.float32)

    colored = colorize(d)
    if isinstance(colored, torch.Tensor):
        return to_pil_image(colored.detach().cpu())
    elif isinstance(colored, np.ndarray):
        arr = colored
        if arr.dtype != np.uint8:
            arr = np.clip(arr, 0, 255).astype(np.uint8)
        if arr.ndim == 3 and arr.shape[0] in (1, 3) and arr.shape[2] not in (1, 3):
            arr = np.transpose(arr, (1, 2, 0))
        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)
        return Image.fromarray(arr)
    else:
        raise TypeError(f"Unexpected type from colorize(): {type(colored)}")


# DEVICE CHECK
device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "This script expects CUDA."


# PRECOMPUTE CONSTANTS THAT DON'T DEPEND ON IMAGE SIZE
azi = torch.tensor(np.deg2rad(azi_deg), device=device)
z_up = torch.tensor([[0., 0., 1.]], device=device)

src_view_mtx = camera_lookat(
    torch.tensor([[0., 0., 0.]], device=device),
    torch.tensor([[-1., 0., 0.]], device=device),
    z_up
)

# fovy from focal length (same as your original)
try:
    fovy = np.deg2rad(fovy_deg)  # noqa: F821
except Exception:
    fovy = focal_length_to_fov(focal_length_mm, 24.0)
fovy = torch.ones(1, device=device) * fovy


# LOG FILE (recommended)
log_path = join(out_dir, "genwarp_params_log.csv")
with open(log_path, "w", encoding="utf-8") as f:
    f.write("idx,filename,orig_w,orig_h,proc_w,proc_h,ele_deg,radius,out_file\n")


# MAIN LOOP
image_files = get_first_n_images(image_dir, n=100)
print(f"Found {len(image_files)} images (taking up to 100).")
print(f"Logging params to: {log_path}")

# NOTE: assumes you already created:
#   - zoedepth (with .infer)
#   - genwarp_nvs (GenWarp instance you call like genwarp_nvs(...))
#
# Example (from repo):
#   from genwarp import GenWarp
#   genwarp_nvs = GenWarp(cfg=genwarp_cfg)
#   zoedepth = torch.hub.load(...).to("cuda")

for idx, image_file in enumerate(image_files, start=1):
    name = splitext(basename(image_file))[0]

    # ---- RANDOM VIEW PARAMETERS ----
    ele_deg = random.choice(ELE_OPTIONS)
    radius = random.choice(RADIUS_OPTIONS)
    ele = torch.tensor(np.deg2rad(ele_deg), device=device)

    out_file = f"{idx:02d}_{name}_ele{ele_deg}_r{radius:.2f}.jpg"
    out_path = join(out_dir, out_file)

    try:
        print(f"\n[{idx:02d}] Processing: {image_file}")
        print(f"     Using ele_deg={ele_deg}, radius={radius}")

        # 1) LOAD (KEEP ORIGINAL SIZE; NO CROP, NO RESIZE)
        pil = Image.open(image_file).convert("RGB")
        orig_w, orig_h = pil.size  # e.g., 752x480 (W,H)

        # Optional safety: pad to multiple of 8 (keeps structure; no crop)
        # If your images are already multiples of 8 (752 and 480 are), this does nothing.
        pil_proc, pad = pad_to_multiple(pil, mult=8, fill=(0, 0, 0))
        proc_w, proc_h = pil_proc.size

        # 2) TENSOR (GPU)
        src_image = to_tensor(pil_proc)[None].to(device, non_blocking=True)  # float32 BCHW

        # 3) DEPTH
        with torch.no_grad():
            src_depth = zoedepth.infer(src_image)

        if isinstance(src_depth, np.ndarray):
            src_depth = torch.from_numpy(src_depth)

        if src_depth.device.type != "cuda":
            src_depth = src_depth.to(device)

        # 4) PROJECTION MATRIX WITH CORRECT ASPECT
        aspect_wh = float(proc_w) / float(proc_h)
        src_proj_mtx = get_projection_matrix(
            fovy=fovy,
            aspect_wh=aspect_wh,
            near=near,
            far=far
        ).to(device)
        tar_proj_mtx = src_proj_mtx

        # 5) (OPTIONAL) HALF PRECISION LIKE YOUR ORIGINAL
        src_image = src_image.half()
        src_depth = src_depth.half()

        # 6) CAMERA
        mean_depth = src_depth.mean(dim=(2, 3)).squeeze(1)  # (B,)
        eye = sph2cart(azi, ele, mean_depth + radius).float()
        at = F.pad(-mean_depth[:, None], (0, 2), mode="constant", value=0)

        tar_view_mtx = camera_lookat(eye + at, at, z_up)
        rel_view_mtx = (tar_view_mtx @ torch.linalg.inv(src_view_mtx.float())).to(src_image)

        # 7) GENWARP
        with torch.no_grad():
            renders = genwarp_nvs(
                src_image=src_image,
                src_depth=src_depth,
                rel_view_mtx=rel_view_mtx,
                src_proj_mtx=src_proj_mtx.to(src_image),
                tar_proj_mtx=tar_proj_mtx.to(src_image),
            )

        synthesized = renders["synthesized"]  # BCHW, same H/W as input tensor

        # 8) SAVE: remove padding (if any) to restore EXACT original W×H 
        synth_pil = to_pil_image(synthesized[0].float().cpu())  # PIL is (W,H) = (proc_w, proc_h)
        synth_pil = unpad_pil(synth_pil, pad)                   # back to (orig_w, orig_h)

        # (Safety) If something slightly off, force final size:
        if synth_pil.size != (orig_w, orig_h):
            synth_pil = synth_pil.resize((orig_w, orig_h), Image.BICUBIC)

        synth_pil.save(out_path, quality=95)

        # 9) LOG PARAMS
        with open(log_path, "a", encoding="utf-8") as f:
            f.write(f"{idx},{basename(image_file)},{orig_w},{orig_h},{proc_w},{proc_h},{ele_deg},{radius:.2f},{out_file}\n")

        print(f"[{idx:02d}/{len(image_files):02d}] Saved: {out_path}")
        print(f"     Output size: {synth_pil.size} (should be {(orig_w, orig_h)})")

    except BaseException as e:
        print(f"[{idx:02d}/{len(image_files):02d}] FAILED on {image_file}")
        print("Exception type:", type(e))
        print("Exception repr:", repr(e))
        traceback.print_exc()

    finally:
        cleanup_vars(
            locals(),
            [
                "pil", "pil_proc", "src_image", "src_depth", "renders", "synthesized",
                "rel_view_mtx", "tar_view_mtx", "src_proj_mtx", "tar_proj_mtx",
                "synth_pil"
            ]
        )

print("\nDone.")
print(f"Parameter log saved at: {log_path}")


Found 100 images (taking up to 100).
Logging params to: ../es_100_ref_high/genwarp_params_log.csv

[01] Processing: ../es_100_ref/1.jpg
     Using ele_deg=12, radius=0.31
[01/100] Saved: ../es_100_ref_high/01_1_ele12_r0.31.jpg
     Output size: (2160, 2160) (should be (2160, 2160))

[02] Processing: ../es_100_ref/101.jpg
     Using ele_deg=15, radius=0.31
[02/100] Saved: ../es_100_ref_high/02_101_ele15_r0.31.jpg
     Output size: (2160, 2160) (should be (2160, 2160))

[03] Processing: ../es_100_ref/103.jpg
     Using ele_deg=14, radius=0.31
[03/100] Saved: ../es_100_ref_high/03_103_ele14_r0.31.jpg
     Output size: (2160, 2160) (should be (2160, 2160))

[04] Processing: ../es_100_ref/104.jpg
     Using ele_deg=12, radius=0.4
[04/100] Saved: ../es_100_ref_high/04_104_ele12_r0.40.jpg
     Output size: (2160, 2160) (should be (2160, 2160))

[05] Processing: ../es_100_ref/105.jpg
     Using ele_deg=19, radius=0.31
[05/100] Saved: ../es_100_ref_high/05_105_ele19_r0.31.jpg
     Output size: 